# 04 — BiomedCLIP Alignment & Multi-task Learning

**Phases 7 and 8.** This notebook does four things, in increasing order of
cost and decreasing order of reliability:

1. **Extract frozen BiomedCLIP embeddings** for every frame and every report
   section. Cheap, deterministic, and reused by notebooks 05 and 06 — run this
   even if you skip everything below.
2. **Zero-shot classification** with prompt ensembling. No training at all.
   On 259 studies this is a real result: if zero-shot beats your fine-tuned
   CNN on rare classes, that belongs in the paper.
3. **Linear probe** on frozen features. The strongest honest baseline for a
   dataset this size, and the one reviewers will ask for.
4. **Multi-task fine-tuning**: shared encoder, classification head +
   report-generation head, with a cross-modal contrastive term.

### Section-aware encoding (proposal contribution #2)

Findings and impression are different registers: findings are long, organ-by-
organ, and mostly negative ("no focal lesion is seen"); the impression is one
or two short diagnostic clauses. Encoding the concatenation lets 140 words of
boilerplate drown 8 words of diagnosis. We encode the two sections separately
and align the **image** with both under separate temperatures. The ablation in
notebook 06 measures whether this actually helps — report it either way.

### Expectation management

With 259 studies, a fine-tuned generation head will produce fluent
radiology-shaped text that is frequently wrong about the specific finding.
That is not a bug in your code, it is the dataset size. It is also the
motivation for notebook 05: retrieval grounds generation in real reports from
similar cases instead of asking a decoder to memorise a vocabulary it has
seen a few hundred times.

**Requirements:** GPU + internet on (Settings → Internet → On) for the Hub.

In [1]:
import os
import sys
import re
import json
import time
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

SEED = 42
np.random.seed(SEED)


def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
EMB = ART / "embeddings"
for d in (ART, FIG, EMB):
    d.mkdir(parents=True, exist_ok=True)


# Artifacts are resolved one file at a time, because on Kaggle each notebook's
# output is a separate dataset: master_split.csv comes from notebook 02 while
# label_tiers.json and ontology.json come from notebook 00. Attach whichever
# outputs you need — this searches all of them.
PRODUCED_BY = {
    "master.csv": "00", "ontology.json": "00", "label_tiers.json": "00",
    "label_audit.csv": "00", "labels_multihot.csv": "00 (refreshed by 02)",
    "master_split.csv": "02", "frames.csv": "02", "preprocess_config.json": "02",
}


def _rank(p):
    """Prefer the most downstream notebook when a file exists in several inputs."""
    nums = re.findall(r"[/\\](\d{2})[-_]", str(p))
    return max((int(n) for n in nums), default=-1)


def find_file(fname, required=True):
    if (ART / fname).exists():
        return ART / fname
    hits = sorted(Path("/kaggle/input").rglob(fname), key=_rank, reverse=True)
    if hits:
        return hits[0]
    if not required:
        return None
    raise FileNotFoundError(
        f"'{fname}' not found in /kaggle/working/artifacts or any attached dataset.\n"
        f"It is produced by notebook {PRODUCED_BY.get(fname, '00-02')}. "
        "Use Add Data -> Notebook Output to attach that notebook's output, "
        "then re-run this cell."
    )


for _f in ("master_split.csv", "frames.csv", "label_tiers.json", "labels_multihot.csv"):
    print(f"  {_f:24s} <- {find_file(_f).parent}")

df = pd.read_csv(find_file("master_split.csv"))
frames = pd.read_csv(find_file("frames.csv"))
tiers = json.loads(find_file("label_tiers.json").read_text())
Ydf = pd.read_csv(find_file("labels_multihot.csv"))

TARGETS = [t for t in tiers["trainable"] + tiers["marginal"] if t in Ydf.columns]
Y = Ydf[TARGETS].values.astype(np.float32)
folds = df["fold"].values
study_ids = df["study_id"].values
print(f"{len(df)} studies | {len(frames)} frames | {len(TARGETS)} targets")

  master_split.csv         <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  frames.csv               <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  label_tiers.json         <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
  labels_multihot.csv      <- /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/artifacts
259 studies | 494 frames | 13 targets


### Re-anchor frame paths

`frames.csv` stores the absolute paths that existed **while notebook 02 was
running** — `/kaggle/working/frames/...`. In this session those frames live
under `/kaggle/input/<notebook-02-output>/frames/` instead, so every path in
the file is stale.

This matters more than a normal path bug: `cv2.imread` returns `None` for a
missing file rather than raising, the loader substitutes a black image, and
training completes normally on blank inputs. You get plausible-looking loss
curves and meaningless results. The cell below re-anchors the paths and then
asserts the frames actually load.

In [2]:
def _extract_frames_archive():
    """Fall back to frames.zip when the loose frames/ directory is unavailable."""
    import zipfile
    hits = sorted(Path("/kaggle/input").rglob("frames.zip"))
    if not hits:
        return None
    dest = Path("/kaggle/working/frames")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(hits[0]) as zf:
        zf.extractall(dest)
        n = len(zf.namelist())
    print(f"Extracted {n} frames from {hits[0]} -> {dest}")
    return dest


def resolve_frames_root(frames_df):
    """Find the directory that actually holds the frames in this session."""
    sample = Path(str(frames_df["frame_path"].iloc[0]))
    if sample.exists():
        return None                                   # paths are already valid
    rel = Path(sample.parent.name) / sample.name      # e.g. R1/R1_p00.jpg
    candidates = [Path("/kaggle/working/frames")]
    for depth in ("*/frames", "*/*/frames", "*/frames/*"):
        candidates += sorted(Path("/kaggle/input").glob(depth))
    for c in candidates:
        if (c / rel).exists():
            return c
    unzipped = _extract_frames_archive()              # last resort: frames.zip
    if unzipped is not None and (unzipped / rel).exists():
        return unzipped
    raise FileNotFoundError(
        f"Frames not found. frames.csv points at {sample}, which does not exist "
        "here, and no attached dataset contains a 'frames/' folder or a "
        "'frames.zip' holding that file.\n"
        "Re-run notebook 02 (it now writes /kaggle/working/frames.zip), save the "
        "version, and attach that output here."
    )


_root = resolve_frames_root(frames)
if _root is not None:
    frames["frame_path"] = [
        str(_root / Path(p).parent.name / Path(p).name) for p in frames["frame_path"]]
    print(f"Re-anchored {len(frames)} frame paths to {_root}")
else:
    print("Frame paths already valid.")

_missing = [p for p in frames["frame_path"] if not os.path.exists(p)]
assert not _missing, (
    f"{len(_missing)} of {len(frames)} frame files are missing, e.g. {_missing[:3]}. "
    "Re-save notebook 02's output including the frames/ directory."
)

# Decode one frame and confirm it is not blank — a file can exist and still be
# unreadable, and a silently black dataset is the failure this guards against.
import cv2 as _cv2
_probe = _cv2.imread(frames["frame_path"].iloc[0])
assert _probe is not None, f"cv2 could not decode {frames['frame_path'].iloc[0]}"
assert _probe.std() > 1.0, (
    "First frame decodes to a near-constant image — the frames are blank. "
    "Re-run notebook 02's panel decomposition."
)
print(f"Frames verified: {len(frames)} files, sample {_probe.shape}, "
      f"pixel std {_probe.std():.1f}")

Extracted 494 frames from /kaggle/input/notebooks/ashrafbin/02-preprocessing-splits/frames.zip -> /kaggle/working/frames
Re-anchored 494 frame paths to /kaggle/working/frames
Frames verified: 494 files, sample (224, 224, 3), pixel std 99.3


In [3]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    DEVICE = "cpu"
print("device:", DEVICE)
if HAS_TORCH and DEVICE == "cpu":
    print("WARNING: no GPU detected. Turn on the accelerator in Kaggle settings — "
          "fine-tuning will take hours on CPU.")

try:
    import open_clip  # noqa
except ImportError:
    pipq("open_clip_torch", "transformers")
    try:
        import open_clip  # noqa
    except ImportError:
        open_clip = None

device: cuda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


## 1. Load BiomedCLIP

BiomedCLIP (Zhang et al., 2023) pairs a ViT-B/16 image tower with a
PubMedBERT text tower, trained on 15M biomedical figure-caption pairs. It is
the right starting point: general CLIP has never seen a sector-scan
ultrasound with console annotations.

If the Hub is unreachable, the notebook falls back to a general CLIP
checkpoint so the pipeline still runs — but say which one you used in the
paper, the difference is large.

In [4]:
BIOMEDCLIP = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
FALLBACK = ("ViT-B-16", "laion2b_s34b_b88k")

clip_model = tokenizer = preprocess = None
BACKBONE_NAME = None

if HAS_TORCH and open_clip is not None:
    try:
        clip_model, preprocess = open_clip.create_model_from_pretrained(BIOMEDCLIP)
        tokenizer = open_clip.get_tokenizer(BIOMEDCLIP)
        BACKBONE_NAME = "BiomedCLIP"
    except Exception as e:
        print(f"BiomedCLIP unavailable ({type(e).__name__}) — falling back to open CLIP.")
        try:
            clip_model, _, preprocess = open_clip.create_model_and_transforms(
                FALLBACK[0], pretrained=FALLBACK[1])
            tokenizer = open_clip.get_tokenizer(FALLBACK[0])
            BACKBONE_NAME = f"CLIP-{FALLBACK[0]}"
        except Exception as e2:
            print(f"No CLIP checkpoint available: {e2}")

if clip_model is not None:
    clip_model = clip_model.to(DEVICE).eval()
    EMB_DIM = clip_model.text_projection.shape[-1] if hasattr(clip_model, "text_projection") else 512
    print(f"Loaded {BACKBONE_NAME}")

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Loaded BiomedCLIP


## 2. Frozen embeddings

Frames are embedded individually, then averaged into a study embedding
(L2-normalised before and after — averaging unnormalised CLIP features lets
one high-norm frame dominate the bag).

We keep **both** the per-frame matrix and the study matrix: notebook 05
retrieves at frame level (finer-grained matches) and evaluates at study level.

In [5]:
BATCH = 64


@torch.no_grad()
def embed_images(paths):
    from PIL import Image
    out, failed = [], []
    for i in range(0, len(paths), BATCH):
        batch = []
        for p in paths[i:i + BATCH]:
            try:
                batch.append(preprocess(Image.open(p).convert("RGB")))
            except Exception as e:
                failed.append((p, type(e).__name__))
                batch.append(torch.zeros(3, 224, 224))
        x = torch.stack(batch).to(DEVICE)
        f = clip_model.encode_image(x)
        out.append(F.normalize(f, dim=-1).cpu().numpy())
        if (i // BATCH) % 10 == 0:
            print(f"    {i}/{len(paths)}", end="\r")
    if failed:
        # Zero vectors here would quietly poison retrieval in notebook 05.
        raise RuntimeError(
            f"{len(failed)} of {len(paths)} images failed to load, e.g. "
            f"{failed[:3]}. Fix the paths before embedding."
        )
    return np.concatenate(out).astype(np.float32)


@torch.no_grad()
def embed_texts(texts, max_len=256):
    out = []
    for i in range(0, len(texts), BATCH):
        toks = tokenizer(list(texts[i:i + BATCH]), context_length=max_len).to(DEVICE)
        f = clip_model.encode_text(toks)
        out.append(F.normalize(f, dim=-1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


if clip_model is not None:
    t0 = time.time()
    frame_paths = frames["frame_path"].tolist()
    Zframe = embed_images(frame_paths)
    np.save(EMB / "frame_embeddings.npy", Zframe)
    frames[["study_id", "frame_id", "frame_path"]].to_csv(EMB / "frame_index.csv", index=False)

    # Study embedding = renormalised mean of its frame embeddings.
    idx_by_study = frames.groupby("study_id").indices
    Zstudy = np.zeros((len(df), Zframe.shape[1]), np.float32)
    for i, sid in enumerate(study_ids):
        ix = idx_by_study.get(sid, [])
        if len(ix):
            v = Zframe[list(ix)].mean(0)
            Zstudy[i] = v / (np.linalg.norm(v) + 1e-8)
    np.save(EMB / "study_image_embeddings.npy", Zstudy)

    # Section-aware text embeddings.
    Zfind = embed_texts(df["findings_proc"].fillna("").values)
    Zimp = embed_texts(df["impression"].fillna("").values)
    Zfull = embed_texts(df["report_proc"].fillna("").values)
    np.save(EMB / "text_findings.npy", Zfind)
    np.save(EMB / "text_impression.npy", Zimp)
    np.save(EMB / "text_full.npy", Zfull)
    print(f"\nEmbedded in {time.time()-t0:.0f}s | frames {Zframe.shape} | studies {Zstudy.shape}")
else:
    Zframe = Zstudy = Zfind = Zimp = Zfull = None
    print("No CLIP model — embedding step skipped.")

    0/494
Embedded in 17s | frames (494, 512) | studies (259, 512)


### Sanity check: is the pretrained space already meaningful?

Before training anything, check whether image embeddings of studies sharing a
label are closer to each other than to random studies. If the ratio is ~1.0,
BiomedCLIP's image tower sees nothing useful in these frames and you should
expect fine-tuning to do all the work — worth knowing before you spend GPU
hours.

In [6]:
if Zstudy is not None:
    S = Zstudy @ Zstudy.T
    np.fill_diagonal(S, np.nan)
    rows = []
    for j, lab in enumerate(TARGETS):
        pos = np.where(Y[:, j] == 1)[0]
        if len(pos) < 3:
            continue
        within = np.nanmean(S[np.ix_(pos, pos)])
        between = np.nanmean(S[np.ix_(pos, np.where(Y[:, j] == 0)[0])])
        rows.append({"label": lab, "n": len(pos), "within": within,
                     "between": between, "gap": within - between})
    sep = pd.DataFrame(rows).sort_values("gap", ascending=False)
    sep[["within", "between", "gap"]] = sep[["within", "between", "gap"]].round(4)
    sep.to_csv(ART / "embedding_separability.csv", index=False)
    print(sep.to_string(index=False))
    print(f"\nMean within-class gap: {sep['gap'].mean():+.4f} "
          "(>0 means the frozen space already separates that class)")

                  label  n  within  between     gap
         Cholelithiasis  5  0.8614   0.7933  0.0681
Early pregnancy failure 14  0.8226   0.7550  0.0677
             Renal cyst  6  0.8336   0.7878  0.0457
        Significant PVR 10  0.8149   0.7740  0.0409
           Hepatomegaly 15  0.7994   0.7621  0.0373
            Fatty liver 28  0.7900   0.7544  0.0356
               Cystitis 21  0.7995   0.7685  0.0310
           Ovarian cyst  5  0.7877   0.7703  0.0174
           Bulky uterus 16  0.7712   0.7538  0.0174
       Normal pregnancy 42  0.7300   0.7204  0.0096
           Normal study 84  0.7464   0.7437  0.0027
      Pelvic collection  7  0.6958   0.7178 -0.0220
  Thickened endometrium  6  0.6777   0.7148 -0.0370

Mean within-class gap: +0.0242 (>0 means the frozen space already separates that class)


## 3. Zero-shot classification with prompt ensembling

One prompt per class is noisy; an ensemble of paraphrases averaged in
embedding space is markedly more stable. Write prompts in the language of
the reports, not textbook language — "fatty change in liver" matches these
reports better than "hepatic steatosis".

In [7]:
PROMPTS = {
    "Normal study": [
        "ultrasound showing normal findings",
        "normal sonographic study, no abnormality detected",
        "unremarkable abdominal ultrasound",
    ],
    "Normal pregnancy": [
        "ultrasound of a live intrauterine pregnancy",
        "obstetric ultrasound showing a gestational sac with fetal heart activity",
        "single alive intrauterine pregnancy",
    ],
    "Fatty liver": [
        "ultrasound showing fatty change in the liver",
        "increased hepatic echogenicity consistent with hepatic steatosis",
        "bright liver parenchyma on ultrasound",
    ],
    "Cystitis": [
        "ultrasound showing thickened irregular urinary bladder wall",
        "sonographic features of cystitis",
        "bladder wall thickening with sediment in urine",
    ],
    "Bulky uterus": [
        "ultrasound showing a bulky enlarged uterus",
        "enlarged uterus on pelvic ultrasound",
    ],
    "Hepatomegaly": [
        "ultrasound showing an enlarged liver",
        "hepatomegaly on abdominal ultrasound",
    ],
    "Early pregnancy failure": [
        "ultrasound showing early pregnancy failure, missed abortion",
        "gestational sac without fetal cardiac activity",
        "blighted ovum on transvaginal ultrasound",
    ],
    "Significant PVR": [
        "ultrasound showing significant post void residual urine",
        "incomplete bladder emptying on ultrasound",
    ],
    "Pelvic collection": [
        "ultrasound showing fluid collection in the pouch of Douglas",
        "pelvic fluid collection on ultrasound",
    ],
    "Renal cyst": [
        "ultrasound showing a renal cortical cyst",
        "anechoic cyst in the kidney on ultrasound",
    ],
    "Thickened endometrium": [
        "ultrasound showing thickened endometrium",
        "increased endometrial thickness on pelvic ultrasound",
    ],
    "Cholelithiasis": [
        "ultrasound showing gallstones with posterior acoustic shadowing",
        "echogenic structure in the gallbladder lumen casting a shadow",
    ],
    "Ovarian cyst": [
        "ultrasound showing an ovarian cyst",
        "cystic lesion in the ovary with internal echoes",
        "adnexal cystic mass on ultrasound",
    ],
}

if clip_model is not None:
    missing_prompts = [t for t in TARGETS if t not in PROMPTS]
    if missing_prompts:
        print("No prompts for:", missing_prompts, "— add them before reporting zero-shot.")
    cls_vecs = []
    for t in TARGETS:
        ps = PROMPTS.get(t, [t.lower()])
        v = embed_texts(np.array(ps)).mean(0)
        cls_vecs.append(v / (np.linalg.norm(v) + 1e-8))
    Wzs = np.stack(cls_vecs)                       # n_class x d
    logits_zs = Zstudy @ Wzs.T
    # Map cosine similarity to [0,1] per class so thresholds are comparable.
    zs_prob = ((logits_zs - logits_zs.min(0))
                / (np.ptp(logits_zs, axis=0) + 1e-8))
    np.save(ART / "oof_zeroshot.npy", zs_prob.astype(np.float32))

    from sklearn.metrics import roc_auc_score
    aucs = [(t, roc_auc_score(Y[:, j], zs_prob[:, j]))
            for j, t in enumerate(TARGETS) if 0 < Y[:, j].sum() < len(Y)]
    zdf = pd.DataFrame(aucs, columns=["label", "zeroshot_AUC"]).sort_values(
        "zeroshot_AUC", ascending=False)
    zdf["zeroshot_AUC"] = zdf["zeroshot_AUC"].round(3)
    zdf.to_csv(ART / "zeroshot_auc.csv", index=False)
    print(zdf.to_string(index=False))
    print(f"\nmean zero-shot AUC = {zdf['zeroshot_AUC'].mean():.3f}  (0.5 = chance)")

                  label  zeroshot_AUC
Early pregnancy failure         0.755
             Renal cyst         0.692
           Bulky uterus         0.678
           Normal study         0.640
           Hepatomegaly         0.630
            Fatty liver         0.619
       Normal pregnancy         0.608
  Thickened endometrium         0.588
         Cholelithiasis         0.587
           Ovarian cyst         0.534
               Cystitis         0.528
        Significant PVR         0.492
      Pelvic collection         0.467

mean zero-shot AUC = 0.601  (0.5 = chance)


## 4. Linear probe on frozen features

Logistic regression on frozen embeddings, same folds as notebook 03. This is
the number to beat with fine-tuning; on a few hundred images it very often
is not beaten, which is itself a result worth stating.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def probe(Z, tag, C=1.0):
    oof = np.zeros_like(Y)
    for f in sorted(set(folds)):
        tr, te = folds != f, folds == f
        for j in range(len(TARGETS)):
            if Y[tr, j].sum() < 2:
                oof[te, j] = Y[tr, j].mean()
                continue
            clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
            clf.fit(Z[tr], Y[tr, j])
            oof[te, j] = clf.predict_proba(Z[te])[:, 1]
    np.save(ART / f"oof_{tag}.npy", oof)
    micro = f1_score(Y, (oof >= 0.5).astype(int), average="micro", zero_division=0)
    macro = f1_score(Y, (oof >= 0.5).astype(int), average="macro", zero_division=0)
    print(f"  {tag:24s} microF1={micro:.3f} macroF1={macro:.3f}")
    return oof


if Zstudy is not None:
    print("Linear probes (threshold 0.5, notebook 06 re-tunes properly):")
    oof_probe_img = probe(Zstudy, "probe_image")
    oof_probe_txt = probe(Zfind, "probe_text_findings")
    oof_probe_cat = probe(np.hstack([Zstudy, Zfind]), "probe_concat")

Linear probes (threshold 0.5, notebook 06 re-tunes properly):
  probe_image              microF1=0.305 macroF1=0.223
  probe_text_findings      microF1=0.321 macroF1=0.272
  probe_concat             microF1=0.379 macroF1=0.278


## 5. Multi-task model

```
           frames ──► BiomedCLIP image tower ──► attention pooling ──► z_img
                                                         │
                       ┌─────────────────────────────────┼──────────────────┐
                       ▼                                 ▼                  ▼
             classification head              generation head        contrastive
             (multi-label BCE)            (prefix-conditioned LM)   (image ↔ text)
```

Total loss = `λ_cls · L_cls + λ_gen · L_gen + λ_con · L_con`.

Three practical choices that matter on a small dataset:

- **Freeze most of the image tower.** Unfreezing all of ViT-B/16 on 259
  studies destroys the pretrained features within two epochs. We unfreeze the
  last `N_UNFROZEN` blocks only.
- **The decoder is small.** `distilgpt2` conditioned on a projected image
  prefix. A larger decoder memorises the training reports verbatim.
- **Contrastive term uses in-batch negatives.** With batch size 8 the
  contrastive signal is weak; that is expected, and the ablation will show it.

In [9]:
LAMBDA_CLS = 1.0
LAMBDA_GEN = 1.0
LAMBDA_CON = 0.3
N_UNFROZEN = 3
PREFIX_LEN = 8
EPOCHS_MT = 20
LR_MT = 2e-5
LR_HEAD_MT = 1e-3
BATCH_MT = 8
MAX_FRAMES = 6
GEN_MAXLEN = 160
DECODER = "distilgpt2"
RUN_MULTITASK = True

# The generation target is the impression, not the whole report. Generating 150
# words of organ-by-organ boilerplate inflates BLEU while saying nothing; the
# impression is the clinically meaningful part and the honest target.
GEN_TARGET_COL = "impression"

if RUN_MULTITASK and clip_model is not None:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        HAS_HF = True
    except ImportError:
        pipq("transformers")
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            HAS_HF = True
        except ImportError:
            HAS_HF = False
            print("transformers unavailable — multi-task section skipped.")
else:
    HAS_HF = False

In [10]:
if RUN_MULTITASK and HAS_HF and clip_model is not None:
    from PIL import Image

    frames_by_study = frames.groupby("study_id")["frame_path"].apply(list).to_dict()
    gen_texts = df[GEN_TARGET_COL].fillna("no abnormality detected").values
    gtok = AutoTokenizer.from_pretrained(DECODER)
    gtok.pad_token = gtok.eos_token

    class MTDataset(Dataset):
        def __init__(self, idx, train):
            self.idx, self.train = idx, train

        def __len__(self):
            return len(self.idx)

        def __getitem__(self, k):
            i = self.idx[k]
            paths = frames_by_study.get(study_ids[i], [])[:MAX_FRAMES]
            imgs = []
            for p in paths:
                try:
                    imgs.append(preprocess(Image.open(p).convert("RGB")))
                except Exception:
                    imgs.append(torch.zeros(3, 224, 224))
            if not imgs:
                imgs = [torch.zeros(3, 224, 224)]
            x = torch.stack(imgs)
            enc = gtok(gen_texts[i], truncation=True, max_length=GEN_MAXLEN,
                       padding="max_length", return_tensors="pt")
            return (x, torch.tensor(Y[i]), enc["input_ids"][0],
                    enc["attention_mask"][0], i)

    def mt_collate(batch):
        n = max(b[0].shape[0] for b in batch)
        xs, ms = [], []
        for b in batch:
            x = b[0]
            k = x.shape[0]
            if k < n:
                x = torch.cat([x, torch.zeros(n - k, *x.shape[1:])], 0)
            m = torch.zeros(n)
            m[:k] = 1
            xs.append(x)
            ms.append(m)
        return (torch.stack(xs), torch.stack(ms),
                torch.stack([b[1] for b in batch]),
                torch.stack([b[2] for b in batch]),
                torch.stack([b[3] for b in batch]),
                torch.tensor([b[4] for b in batch]))

    class MultiTaskVLM(nn.Module):
        def __init__(self, clip, n_cls, decoder_name=DECODER, prefix_len=PREFIX_LEN):
            super().__init__()
            self.visual = clip.visual
            self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))
            with torch.no_grad():
                d = self.visual(torch.zeros(1, 3, 224, 224).to(
                    next(self.visual.parameters()).device)).shape[-1]
            self.d = d
            self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
            self.cls_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, n_cls))
            self.lm = AutoModelForCausalLM.from_pretrained(decoder_name)
            h = self.lm.config.n_embd
            self.prefix_len = prefix_len
            self.to_prefix = nn.Sequential(
                nn.Linear(d, h * prefix_len // 2), nn.GELU(),
                nn.Linear(h * prefix_len // 2, h * prefix_len))
            self.proj_align = nn.Linear(d, 512)

        def encode_bag(self, x, mask):
            B, N = x.shape[:2]
            h = self.visual(x.flatten(0, 1)).view(B, N, -1)
            a = self.att(h).squeeze(-1).masked_fill(mask == 0, -1e4)
            a = torch.softmax(a, 1)
            return (a.unsqueeze(-1) * h).sum(1), a

        def forward(self, x, mask, input_ids=None, attn=None):
            z, a = self.encode_bag(x, mask)
            logits_cls = self.cls_head(z)
            out = {"cls": logits_cls, "z": z, "att": a,
                   "align": F.normalize(self.proj_align(z), dim=-1)}
            if input_ids is not None:
                B = z.shape[0]
                pre = self.to_prefix(z).view(B, self.prefix_len, -1)
                emb = self.lm.transformer.wte(input_ids)
                inp = torch.cat([pre, emb], 1)
                am = torch.cat([torch.ones(B, self.prefix_len, device=attn.device), attn], 1)
                labels = torch.cat([
                    torch.full((B, self.prefix_len), -100, device=input_ids.device,
                               dtype=input_ids.dtype),
                    input_ids.masked_fill(attn == 0, -100)], 1)
                out["gen_loss"] = self.lm(inputs_embeds=inp, attention_mask=am,
                                          labels=labels).loss
            return out

        @torch.no_grad()
        def generate(self, x, mask, max_new_tokens=60):
            z, _ = self.encode_bag(x, mask)
            B = z.shape[0]
            pre = self.to_prefix(z).view(B, self.prefix_len, -1)
            bos = torch.full((B, 1), gtok.bos_token_id or gtok.eos_token_id,
                             device=z.device, dtype=torch.long)
            inp = torch.cat([pre, self.lm.transformer.wte(bos)], 1)
            out = self.lm.generate(
                inputs_embeds=inp,
                attention_mask=torch.ones(inp.shape[:2], device=z.device),
                max_new_tokens=max_new_tokens, num_beams=4, no_repeat_ngram_size=3,
                pad_token_id=gtok.eos_token_id)
            return gtok.batch_decode(out, skip_special_tokens=True)

    def freeze_visual(model, n_unfrozen=N_UNFROZEN):
        for p in model.visual.parameters():
            p.requires_grad = False
        blocks = None
        for attr in ("transformer", "trunk"):
            obj = getattr(model.visual, attr, None)
            if obj is not None:
                blocks = getattr(obj, "resblocks", None) or getattr(obj, "blocks", None)
                if blocks is not None:
                    break
        if blocks is None:
            print("  (could not locate transformer blocks — visual tower fully frozen)")
            return 0
        for blk in blocks[-n_unfrozen:]:
            for p in blk.parameters():
                p.requires_grad = True
        return sum(p.numel() for p in model.visual.parameters() if p.requires_grad)

    def contrastive(img_e, txt_e, scale):
        logits = scale * img_e @ txt_e.T
        tgt = torch.arange(len(logits), device=logits.device)
        return 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt))

    Ztxt_t = torch.tensor(Zimp) if Zimp is not None else None

    def train_multitask(tag="multitask", lam_cls=LAMBDA_CLS, lam_gen=LAMBDA_GEN,
                        lam_con=LAMBDA_CON, epochs=EPOCHS_MT):
        oof = np.zeros_like(Y)
        gen_out = {}
        pos_w = torch.tensor(((len(Y) - Y.sum(0)) / np.maximum(Y.sum(0), 1)).clip(1, 15),
                             dtype=torch.float32, device=DEVICE)
        for f in sorted(set(folds)):
            tr = np.where(folds != f)[0]
            te = np.where(folds == f)[0]
            model = MultiTaskVLM(clip_model, len(TARGETS)).to(DEVICE)
            n_tr = freeze_visual(model)
            head = [p for n, p in model.named_parameters()
                    if not n.startswith("visual.") and p.requires_grad]
            vis = [p for n, p in model.named_parameters()
                   if n.startswith("visual.") and p.requires_grad]
            opt = torch.optim.AdamW([{"params": vis, "lr": LR_MT},
                                     {"params": head, "lr": LR_HEAD_MT}], weight_decay=0.01)
            dl = DataLoader(MTDataset(tr, True), batch_size=BATCH_MT, shuffle=True,
                            collate_fn=mt_collate, num_workers=2)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(dl))
            bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
            model.train()
            for ep in range(epochs):
                agg = np.zeros(3)
                for x, m, y, ids, am, bidx in dl:
                    x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
                    ids, am = ids.to(DEVICE), am.to(DEVICE)
                    out = model(x, m, ids, am)
                    l_cls = bce(out["cls"], y)
                    l_gen = out["gen_loss"]
                    if Ztxt_t is not None and lam_con > 0:
                        te_ = Ztxt_t[bidx].to(DEVICE)
                        if te_.shape[-1] != out["align"].shape[-1]:
                            te_ = F.pad(te_, (0, out["align"].shape[-1] - te_.shape[-1]))
                        l_con = contrastive(out["align"], F.normalize(te_, dim=-1),
                                            model.logit_scale.exp().clamp(max=100))
                    else:
                        l_con = torch.zeros((), device=DEVICE)
                    loss = lam_cls * l_cls + lam_gen * l_gen + lam_con * l_con
                    opt.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad], 1.0)
                    opt.step()
                    sched.step()
                    agg += [l_cls.item(), l_gen.item(), float(l_con)]
                if (ep + 1) % 5 == 0:
                    a = agg / len(dl)
                    print(f"    fold {f} ep {ep+1:3d}  cls {a[0]:.3f}  gen {a[1]:.3f}  con {a[2]:.3f}")
            model.eval()
            probs = []
            with torch.no_grad():
                for x, m, y, ids, am, bidx in DataLoader(
                        MTDataset(te, False), batch_size=8, collate_fn=mt_collate):
                    o = model(x.to(DEVICE), m.to(DEVICE))
                    probs.append(torch.sigmoid(o["cls"]).cpu().numpy())
                    for sid, txt in zip(study_ids[bidx.numpy()],
                                        model.generate(x.to(DEVICE), m.to(DEVICE))):
                        gen_out[sid] = txt.strip()
            oof[te] = np.concatenate(probs)
            torch.save({k: v for k, v in model.state_dict().items()
                        if not k.startswith("lm.")}, ART / f"{tag}_fold{f}.pt")
            del model
            torch.cuda.empty_cache()
            print(f"  fold {f} done ({n_tr/1e6:.1f}M trainable visual params)")
        np.save(ART / f"oof_{tag}.npy", oof)
        pd.DataFrame({"study_id": list(gen_out), "generated": list(gen_out.values())}).to_csv(
            ART / f"generated_{tag}.csv", index=False)
        return oof, gen_out

    t0 = time.time()
    oof_mt, gen_mt = train_multitask()
    print(f"Multi-task training: {(time.time()-t0)/60:.1f} min")
else:
    oof_mt, gen_mt = None, None

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    fold 3 ep   5  cls 0.010  gen 0.601  con 0.146
    fold 3 ep  10  cls 0.004  gen 0.191  con 0.142
    fold 3 ep  15  cls 0.001  gen 0.034  con 0.101
    fold 3 ep  20  cls 0.001  gen 0.005  con 0.086
  fold 3 done (21.3M trainable visual params)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    fold 4 ep   5  cls 0.009  gen 0.662  con 0.173
    fold 4 ep  10  cls 0.003  gen 0.215  con 0.118
    fold 4 ep  15  cls 0.001  gen 0.019  con 0.105
    fold 4 ep  20  cls 0.001  gen 0.003  con 0.089
  fold 4 done (21.3M trainable visual params)
Multi-task training: 36.1 min


## 6. How good is the generated text, really?

Look at examples before you look at BLEU. BLEU on radiology impressions is
dominated by shared boilerplate — a model that always writes "Normal USG
findings." scores respectably. Notebook 05 adds clinical-efficacy metrics
(does the generated text contain the right finding?), which is the number
that actually matters.

In [11]:
if gen_mt:
    ref = dict(zip(df["study_id"], df[GEN_TARGET_COL].fillna("")))
    show = list(gen_mt)[:8]
    for sid in show:
        print(f"--- {sid}")
        print(f"  REF : {ref.get(sid,'')[:110]}")
        print(f"  GEN : {gen_mt[sid][:110]}")
    uniq = len(set(gen_mt.values()))
    print(f"\nDistinct generated impressions: {uniq}/{len(gen_mt)} "
          f"({uniq/len(gen_mt):.0%})")
    print("A low ratio means the decoder collapsed onto one or two safe sentences —\n"
          "the single most common failure mode at this dataset size, and exactly\n"
          "what retrieval augmentation in notebook 05 is meant to fix.")

--- R2
  REF : Right adnexal chocolate cyst.
  GEN : flamed subcutaneous tissue at the site of incision. 2. Bilateral mild enlarged ovaries. 3. Dominant follicle n
--- R9
  REF : Excessive bowel gas shadow present otherwise normal study.
  GEN : case of early pregnancy failure (missed abortion). 2. Mild pelvic collection. Cystitis. 3. Left renal small co
--- R12
  REF : Suggestive of Gross hepatomegaly with Grade-III fatty change.
  GEN : -Ill fatty change in liver. 2. Hepatic metastasis. 3. Enlarged thyroid gland with multiple intramural fibroid.
--- R15
  REF : NORMAL STUDY.
  GEN : ild hepatomegaly with (Gr-I) fatty change. 2. Left renal small cyst with internal calcification. 3. MCC 424 cc
--- R23
  REF : Normal USG findings.
  GEN : + weeks of single USG findings. Cholelithiasis. 2. Hepatic metastasis. 3. Enlarged prostate. 4. No abdominal l
--- R32
  REF : About 08 weeks 04 days of single alive Intrauterine pregnancy with adequate liquor at present.
  GEN : early intrauterine pre

## 7. Results so far

In [12]:
from sklearn.metrics import f1_score, roc_auc_score


def quick(name, p):
    if p is None:
        return None
    pred = (p >= 0.5).astype(int)
    aucs = [roc_auc_score(Y[:, j], p[:, j]) for j in range(len(TARGETS))
            if 0 < Y[:, j].sum() < len(Y)]
    return {"model": name,
            "micro_F1": f1_score(Y, pred, average="micro", zero_division=0),
            "macro_F1": f1_score(Y, pred, average="macro", zero_division=0),
            "macro_AUC": float(np.mean(aucs))}


rows = [r for r in [
    quick("zeroshot", zs_prob if clip_model is not None else None),
    quick("probe_image", oof_probe_img if Zstudy is not None else None),
    quick("probe_text", oof_probe_txt if Zstudy is not None else None),
    quick("probe_concat", oof_probe_cat if Zstudy is not None else None),
    quick("multitask", oof_mt),
] if r]
vlm_df = pd.DataFrame(rows).round(4)
vlm_df.to_csv(ART / "vlm_results.csv", index=False)
print(vlm_df.to_string(index=False))

prev = ART / "baseline_results.csv"
if prev.exists():
    both = pd.concat([pd.read_csv(prev), vlm_df], ignore_index=True)
    both = both.sort_values("macro_F1", ascending=False)
    both.to_csv(ART / "all_results.csv", index=False)
    print("\nCombined with notebook 03 -> all_results.csv")
    print(both[["model", "micro_F1", "macro_F1", "macro_AUC"]].to_string(index=False))

       model  micro_F1  macro_F1  macro_AUC
    zeroshot    0.1583    0.1439     0.6015
 probe_image    0.3046    0.2230     0.6581
  probe_text    0.3208    0.2721     0.7979
probe_concat    0.3791    0.2782     0.7854
   multitask    0.8554    0.7759     0.9375


In [13]:
if Zstudy is not None:
    try:
        from sklearn.manifold import TSNE
        emb2 = TSNE(n_components=2, perplexity=18, init="pca",
                    random_state=SEED).fit_transform(Zstudy)
        prim = df["primary_label_grouped"].fillna("Other").values
        fig, ax = plt.subplots(figsize=(7.5, 6))
        for g in pd.unique(prim):
            k = prim == g
            ax.scatter(emb2[k, 0], emb2[k, 1], s=26, label=g, alpha=0.8)
        ax.legend(fontsize=8, markerscale=1.2, loc="best")
        ax.set_title(f"{BACKBONE_NAME} study embeddings (t-SNE)")
        ax.set_xticks([])
        ax.set_yticks([])
        plt.savefig(FIG / "tsne_image_embeddings.png")
        plt.show()
    except Exception as e:
        print("t-SNE skipped:", e)

In [14]:
meta = {
    "backbone": BACKBONE_NAME,
    "generation_target": GEN_TARGET_COL,
    "lambda": {"cls": LAMBDA_CLS, "gen": LAMBDA_GEN, "con": LAMBDA_CON},
    "n_unfrozen_blocks": N_UNFROZEN,
    "prefix_len": PREFIX_LEN,
    "epochs": EPOCHS_MT,
    "decoder": DECODER,
}
(ART / "vlm_config.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

{
  "backbone": "BiomedCLIP",
  "generation_target": "impression",
  "lambda": {
    "cls": 1.0,
    "gen": 1.0,
    "con": 0.3
  },
  "n_unfrozen_blocks": 3,
  "prefix_len": 8,
  "epochs": 20,
  "decoder": "distilgpt2"
}


## What to carry into the paper

- **Zero-shot AUC table** — a genuinely interesting result, and free. It tells
  you which findings BiomedCLIP already recognises without seeing your data.
- **Embedding separability table** — evidence for or against the claim that
  ultrasound stills carry the signal at all. Put it in the discussion.
- **Linear probe vs fine-tuned** — if the probe wins, say so. "Fine-tuning a
  VLM on 259 studies underperforms a linear probe on frozen features" is a
  useful, citable, honest finding for the small-data medical imaging
  literature.
- Embeddings in `artifacts/embeddings/` are the input to notebook 05.

Next: **05 — retrieval-augmented report generation.**